In [9]:
import pandas as pd
import numpy as np

# 코호트 데이터 로드 (주문 단위, 재구매 여부/첫구매 지연여부 포함)
cohort_df = pd.read_csv('cohort_df.csv')

# 베이스 파일 로드 (아이템 단위, 상세 피처 포함)
base_df = pd.read_csv('../../../Funnel_Cohort_NG/data/olist_order_item_level.csv')

print("=== cohort_df ===")
print("shape:", cohort_df.shape)
print("컬럼:", cohort_df.columns.tolist())
print()
print("=== base_df ===")
print("shape:", base_df.shape)
print("고유 order_id 수:", base_df['order_id'].nunique())

=== cohort_df ===
shape: (99441, 23)
컬럼: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_state', 'review_score', 't_approve_d', 't_carrier_d', 't_delivery_d', 't_total_d', 'delay_days', 'is_delayed', 'is_valid_funnel', 'order_month', 'first_purchase_month', 'cohort_index', 'first_order_is_delayed', 'is_repurchase']

=== base_df ===
shape: (112650, 66)
고유 order_id 수: 98666


# 재구매 예측 모델 (로지스틱 회귀)

## 목적
첫 구매 시점의 배송/리뷰/결제 피처로 재구매 여부를 예측하고,
재구매 가능성이 높은/낮은 고객의 페르소나를 도출한다.

## 데이터 소스
- `cohort_df.csv`: 주문 단위, 재구매 여부(`is_repurchase`), 첫구매 지연여부(`first_order_is_delayed`) 포함
- `olist_order_item_level.csv`: 아이템 단위, 배송/리뷰/결제 상세 피처 포함

## Step 1. 타겟 변수(재구매 여부) + 첫 주문 추출

- `cohort_df`의 `is_repurchase`는 **주문 단위** (cohort_index>0이면 1)
- → 고객 단위로 "한 번이라도 재구매했는지"로 재집계
- 첫 주문 = `cohort_index == 0`인 주문 중 가장 빠른 주문
  (한 고객이 첫 달에 여러 번 주문한 경우 대비)

In [35]:
# 고객별 재구매 여부 (customer-level target)
# cohort_df의 is_repurchase는 '주문' 단위 (cohort_index>0이면 1)
# → 고객 단위로 "한 번이라도 재구매했는지" 로 다시 집계
customer_repurchase = cohort_df.groupby('customer_unique_id')['is_repurchase'].max().reset_index()
customer_repurchase.columns = ['customer_unique_id', 'is_repurchase_customer']

print("고객 수:", len(customer_repurchase))
print("재구매 고객 수:", customer_repurchase['is_repurchase_customer'].sum())
print("재구매율:", round(customer_repurchase['is_repurchase_customer'].mean() * 100, 2), "%")

# 고객별 '첫 주문' 찾기
# cohort_index == 0 인 주문 = 첫 구매월의 주문
# 한 고객이 첫 달에 여러 번 주문했을 가능성 있으니, 그중 가장 빠른 주문을 첫 주문으로 선택
first_orders = (
    cohort_df[cohort_df['cohort_index'] == 0]
    .sort_values('order_purchase_timestamp')
    .groupby('customer_unique_id', as_index=False)
    .first()
)

print("\n첫 주문 수:", len(first_orders))
print("고유 고객 수:", cohort_df['customer_unique_id'].nunique())
print("일치 여부:", len(first_orders) == cohort_df['customer_unique_id'].nunique())

고객 수: 96096
재구매 고객 수: 1797
재구매율: 1.87 %

첫 주문 수: 96096
고유 고객 수: 96096
일치 여부: True


## Step 2. base_df를 '주문 단위'로 집계

`olist_order_item_level.csv`는 **아이템 단위**라 한 주문에 여러 행이 있을 수 있음.
- `review_score_mean`, `is_late`, `customer_region`, `payment_*` 등은 주문당 동일한 값(아이템별 반복) → `first`
- `price`, `freight_value` 등 금액은 아이템별로 다름 → `sum` (주문 전체 금액)

In [36]:
# base_df는 '아이템' 단위 → '주문' 단위로 집계
# - review_score_mean, delivery_days_clean, is_late, customer_region, 
#   payment_value_total, payment_installments_max 등은 주문당 동일한 값(아이템별로 반복됨) → first
# - price, freight_value는 아이템별로 다르므로 → sum (주문 전체 금액)
# - product_category_name_english는 멀티 아이템 주문에서 다를 수 있음 → first (대표 카테고리)

order_level = base_df.groupby('order_id').agg(
    review_score_mean=('review_score_mean', 'first'),
    delivery_days_clean=('delivery_days_clean', 'first'),
    is_late=('is_late', 'first'),
    customer_region=('customer_region', 'first'),
    seller_region=('seller_region', 'first'),
    payment_value_total=('payment_value_total', 'first'),
    payment_installments_max=('payment_installments_max', 'first'),
    payment_type_nunique=('payment_type_nunique', 'first'),
    product_category_name_english=('product_category_name_english', 'first'),
    price=('price', 'sum'),
    freight_value=('freight_value', 'sum'),
    item_total=('item_total', 'sum'),
    n_items=('order_item_id', 'count')
).reset_index()

print("order_level shape:", order_level.shape)
print(order_level.isna().sum())

order_level shape: (98666, 14)
order_id                            0
review_score_mean                 749
delivery_days_clean              3563
is_late                          2190
customer_region                     0
seller_region                       0
payment_value_total                 1
payment_installments_max            1
payment_type_nunique                1
product_category_name_english    1410
price                               0
freight_value                       0
item_total                          0
n_items                             0
dtype: int64


## Step 3. 첫 주문 정보 + 주문 단위 피처 + 타겟 병합

`first_orders`(고객별 첫 주문) + `order_level`(주문별 피처) + `customer_repurchase`(타겟)을
`order_id` / `customer_unique_id` 기준으로 병합하여 모델링용 데이터프레임 `ml_df` 생성.

In [37]:
# 첫 주문 정보(고객, 주문ID, 첫구매 지연여부) + 주문 단위 피처 병합
ml_df = first_orders[['customer_unique_id', 'order_id', 'first_order_is_delayed']].merge(
    order_level, on='order_id', how='left'
)

# 타겟(재구매 여부) 병합
ml_df = ml_df.merge(customer_repurchase, on='customer_unique_id', how='left')

print("ml_df shape:", ml_df.shape)
print()
print("결측치:")
print(ml_df.isna().sum())

ml_df shape: (96096, 17)

결측치:
customer_unique_id                  0
order_id                            0
first_order_is_delayed           2740
review_score_mean                1425
delivery_days_clean              4180
is_late                          2843
customer_region                   708
seller_region                     708
payment_value_total               709
payment_installments_max          709
payment_type_nunique              709
product_category_name_english    2073
price                             708
freight_value                     708
item_total                        708
n_items                           708
is_repurchase_customer              0
dtype: int64


## Step 4. 결측치 패턴 분석

결측치를 단순히 드랍하기 전, **결측 여부가 재구매율과 관련 있는지** 확인.
→ 결측 자체가 의미 있는 정보일 수 있으므로, 무작정 드랍하면 편향이 생길 수 있음.

확인할 것:
1. `base_df`에 자체가 없는 첫 주문 수 (708건 추정)
2. `is_late` / `first_order_is_delayed` 결측 수
3. 결측 여부에 따른 재구매율 차이

In [38]:
# 1) base_df에 자체가 없는 첫 주문 (708건 추정)
missing_order_level = ml_df[ml_df['customer_region'].isna()]
print("base_df에 없는 첫 주문 수:", len(missing_order_level))

# 2) is_late / first_order_is_delayed 결측 - 배송 미완료 의심
print("\nis_late 결측 수:", ml_df['is_late'].isna().sum())
print("first_order_is_delayed 결측 수:", ml_df['first_order_is_delayed'].isna().sum())
print("둘 다 결측인 수:", (ml_df['is_late'].isna() & ml_df['first_order_is_delayed'].isna()).sum())

# 3) 결측 여부에 따른 재구매율 비교 (편향 확인용 - 핵심!)
print("\n=== 재구매율 비교 ===")
print(f"전체: {ml_df['is_repurchase_customer'].mean()*100:.2f}%")
print(f"first_order_is_delayed 결측 그룹: {ml_df[ml_df['first_order_is_delayed'].isna()]['is_repurchase_customer'].mean()*100:.2f}%")
print(f"first_order_is_delayed 존재 그룹: {ml_df[ml_df['first_order_is_delayed'].notna()]['is_repurchase_customer'].mean()*100:.2f}%")

base_df에 없는 첫 주문 수: 708

is_late 결측 수: 2843
first_order_is_delayed 결측 수: 2740
둘 다 결측인 수: 2740

=== 재구매율 비교 ===
전체: 1.87%
first_order_is_delayed 결측 그룹: 0.22%
first_order_is_delayed 존재 그룹: 1.92%


## Step 5. base_df 자체에 없는 708건 상세 확인

708건의 `order_status` 분포와 재구매율을 확인하여,
`first_order_is_delayed` 결측 그룹(2,740건)과 동일한 패턴인지 검증.

In [39]:
# base_df에 자체가 없는 708건 - order_status 확인
status_check = cohort_df[cohort_df['order_id'].isin(missing_order_level['order_id'])]['order_status'].value_counts()
print("base_df에 없는 주문들의 order_status:")
print(status_check)

print(f"\n재구매율: {missing_order_level['is_repurchase_customer'].mean()*100:.2f}%")

base_df에 없는 주문들의 order_status:
order_status
unavailable    583
canceled       117
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

재구매율: 2.54%


## Step 6. 708건의 first_order_is_delayed 분포 확인

708건(부분집합) ⊆ 2,740건(전체집합) 관계가 맞는지 검증.
- 708건 재구매율 2.54% (≈18명) > 2,740건 재구매율 0.22% (≈6명)
- 부분집합의 재구매자가 전체집합보다 많을 수 없으므로, 모순 발생
- → 708건이 2,740건에 완전히 포함되지 않음을 의미

In [40]:
# 708건(base_df 없음) 그룹의 first_order_is_delayed 분포 확인
print(missing_order_level['first_order_is_delayed'].value_counts(dropna=False))

first_order_is_delayed
None     677
False     30
True       1
Name: count, dtype: int64


## Step 7. 모델링 대상 필터링

**핵심 질문**: "첫 배송 경험(지연/정시)이 재구매에 영향을 주는가?"

- `is_late` 결측 = 첫 주문이 정상 배송완료되지 않음 (배송 경험 자체가 없음)
- `customer_region` 결측 = base_df에 주문 자체가 없음
- 두 경우 모두 "배송 경험"을 측정할 수 없으므로 모델링 대상에서 제외

> 참고: `first_order_is_delayed` 기준 결측 그룹(2,740건)은 재구매율 0.22%로 극단적이었으나,
> 이는 `is_late`와 다른 정의 기준이며 샘플(6명)이 너무 작아 노이즈일 가능성 있음.
> `is_late` 기준 제외 그룹은 재구매율이 전체와 유사(아래 확인) → 편향 적음

In [41]:
# 모델링 대상 필터링
# - is_late 결측 = 첫 주문이 정상 배송완료되지 않음 (배송 경험 자체가 없음)
# - customer_region 결측 = base_df에 주문 자체가 없음 (order_items 없는 주문)
# 두 경우 모두 "배송 경험"을 측정할 수 없으므로 모델링 대상에서 제외

excluded = ml_df[ml_df['is_late'].isna() | ml_df['customer_region'].isna()]
included = ml_df[ml_df['is_late'].notna() & ml_df['customer_region'].notna()]

print(f"제외 고객 수: {len(excluded):,} ({len(excluded)/len(ml_df)*100:.2f}%)")
print(f"제외 그룹 재구매율: {excluded['is_repurchase_customer'].mean()*100:.2f}%")
print()
print(f"모델링 대상 고객 수: {len(included):,} ({len(included)/len(ml_df)*100:.2f}%)")
print(f"모델링 대상 재구매율: {included['is_repurchase_customer'].mean()*100:.2f}%")


제외 고객 수: 2,843 (2.96%)
제외 그룹 재구매율: 2.11%

모델링 대상 고객 수: 93,253 (97.04%)
모델링 대상 재구매율: 1.86%


## Step 8. 제외 그룹(2,843건)의 order_status 확인

`is_late` 결측 원인이 '미배송'인지 '이상치(anomaly_flag)'인지 파악.

In [42]:
# 제외 그룹(2,843명)의 order_status 분포 확인
excluded_status = cohort_df[cohort_df['order_id'].isin(excluded['order_id'])]['order_status'].value_counts()
print(excluded_status)

order_status
shipped        1077
unavailable     589
canceled        559
invoiced        308
processing      295
delivered         8
created           5
approved          2
Name: count, dtype: int64


In [43]:
## Step 9. 모델링 대상(included)의 남은 결측치 확인

`is_late`, `customer_region` 기준으로 필터링했지만,
`review_score_mean`, `product_category_name_english` 등은 여전히 결측이 있을 수 있음.
→ 피처별로 결측 처리 방법을 결정하기 위해 확인.

SyntaxError: invalid character '→' (U+2192) (1386037809.py, line 5)

In [ ]:
# 모델링 대상(included)의 결측치 확인
print(included.isna().sum())
print()
print("전체 행 수:", len(included))

customer_unique_id                  0
order_id                            0
first_order_is_delayed              0
review_score_mean                 616
delivery_days_clean              1337
is_late                             0
customer_region                     0
seller_region                       0
payment_value_total                 1
payment_installments_max            1
payment_type_nunique                1
product_category_name_english    1308
price                               0
freight_value                       0
item_total                          0
n_items                             0
is_repurchase_customer              0
dtype: int64

전체 행 수: 93253


## Step 10. 결측치 처리 (1차) - 학습 전 처리 가능한 것들

- `payment_value_total` 등 1건 결측 → 행 드랍 (영향 미미)
- `product_category_name_english` → 'unknown'으로 채움 (상수 대체, 누수 문제 없음)
- `review_score_mean` 결측 자체를 `has_review` 플래그(0/1)로 별도 피처화
  → "리뷰를 안 남긴 고객" 자체가 의미있는 정보일 수 있음

**주의**: `review_score_mean`, `delivery_days_clean`의 실제 결측값 대체(imputation)는
train/test 분리 후 파이프라인 내에서 처리 (train 통계만 사용 → 데이터 누수 방지)

In [ ]:
# 작업용 복사본
ml_data = included.copy()

# 1) payment_* 1건 결측 행 드랍
ml_data = ml_data.dropna(subset=['payment_value_total'])

# 2) product_category_name_english 결측 → 'unknown'
ml_data['product_category_name_english'] = ml_data['product_category_name_english'].fillna('unknown')

# 3) review_score_mean 결측 → has_review 플래그 생성 (1=리뷰 있음, 0=없음)
ml_data['has_review'] = ml_data['review_score_mean'].notna().astype(int)

print("ml_data shape:", ml_data.shape)
print("\n남은 결측치:")
print(ml_data.isna().sum()[ml_data.isna().sum() > 0])
print("\nhas_review 분포:")
print(ml_data['has_review'].value_counts())

ml_data shape: (93252, 18)

남은 결측치:
review_score_mean       616
delivery_days_clean    1337
dtype: int64

has_review 분포:
has_review
1    92636
0      616
Name: count, dtype: int64


## Step 11. 최종 피처 정의

### 제외하는 피처와 이유

| 피처 | 이유 |
|---|---|
| `customer_unique_id`, `order_id` | 식별자, 학습에 불필요 |
| `first_order_is_delayed` | `is_late`와 동일 개념(배송 지연 여부)이지만 다른 파이프라인 정의 → 중복/다중공선성 방지를 위해 `is_late`만 사용 (나경님 분석 전체에서 일관되게 사용된 컬럼) |
| `price`, `item_total` | `payment_value_total`(총 결제금액)과 강한 상관 → 다중공선성. `payment_value_total`을 "주문 금액" 대표 피처로 사용 |

### 최종 피처

**수치형** (스케일링 + 결측치 imputation 필요)
- `delivery_days_clean`, `freight_value`, `payment_value_total`, `payment_installments_max`, `n_items`, `review_score_mean`

**이진형** (0/1, 그대로 사용)
- `is_late`, `has_review`

**범주형** (원-핫 인코딩)
- `customer_region`, `seller_region`, `product_category_name_english`

**타겟**
- `is_repurchase_customer`

In [ ]:
# 피처 그룹 정의
numeric_features = ['delivery_days_clean', 'freight_value', 'payment_value_total', 
                     'payment_installments_max', 'n_items', 'review_score_mean']
binary_features = ['is_late', 'has_review']
categorical_features = ['customer_region', 'seller_region', 'product_category_name_english']

feature_cols = numeric_features + binary_features + categorical_features

X = ml_data[feature_cols].copy()
y = ml_data['is_repurchase_customer'].copy()

print("X shape:", X.shape)
print("y 분포:")
print(y.value_counts())
print(f"양성 비율: {y.mean()*100:.2f}%")

X shape: (93252, 11)
y 분포:
is_repurchase_customer
0    91515
1     1737
Name: count, dtype: int64
양성 비율: 1.86%


## Step 12. Train/Test 분리

재구매율이 1.86%로 극심한 불균형(class imbalance) 상태.
→ `stratify=y`로 train/test 양쪽에 동일한 비율 유지 (80:20 분리)

In [ ]:
from sklearn.model_selection import train_test_split

# stratify=y: 클래스 불균형(1.86%)을 train/test 양쪪에 동일하게 유지
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape, "| 양성 비율:", round(y_train.mean()*100, 2), "%")
print("Test shape:", X_test.shape, "| 양성 비율:", round(y_test.mean()*100, 2), "%")

Train shape: (74601, 11) | 양성 비율: 1.86 %
Test shape: (18651, 11) | 양성 비율: 1.86 %


## Step 13. 전처리 파이프라인 + 로지스틱 회귀

- **수치형**: 결측치는 train 기준 중앙값으로 대체 + 표준화(StandardScaler)
  - imputation을 파이프라인 안에서 처리 → train 통계만 사용, test 데이터 누수 방지
- **범주형**: 원-핫 인코딩 (`handle_unknown='ignore'`로 test에만 있는 카테고리 대응)
- **이진형**: 그대로 통과 (passthrough)
- **클래스 불균형 보정**: `class_weight='balanced'` — 재구매율 1.86%의 극심한 불균형 보정

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

# 수치형: 결측치 → train 중앙값 대체 + RobustScaler (이상치에 강건한 스케일링)
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

# 범주형: 원-핫 인코딩
categorical_pipeline = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# 전체 전처리기
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features),
    ('bin', 'passthrough', binary_features)
])

# 전처리 + 로지스틱 회귀
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])

model.fit(X_train, y_train)
print("학습 완료")

학습 완료


## Step 14. 모델 평가

재구매율 1.86%로 극심한 불균형 → **Accuracy는 의미 없음** (전부 0으로 예측해도 98%대 정확도)
대신 확인할 것:
- **Confusion Matrix**: 재구매 고객을 얼마나 잡아내는지
- **Recall (재구매 클래스)**: 실제 재구매 고객 중 모델이 맞춘 비율 (중요!)
- **Precision (재구매 클래스)**: 모델이 재구매라고 한 고객 중 실제 맞은 비율
- **ROC-AUC**: 전체적인 분류 성능 (0.5=랜덤, 1.0=완벽)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("=== Confusion Matrix ===")
print("        예측 0   예측 1")
cm = confusion_matrix(y_test, y_pred)
print(f"실제 0   {cm[0][0]:6d}  {cm[0][1]:6d}")
print(f"실제 1   {cm[1][0]:6d}  {cm[1][1]:6d}")
print()
print("=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=['비재구매(0)', '재구매(1)']))
print()
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")

=== Confusion Matrix ===
        예측 0   예측 1
실제 0    10510    7794
실제 1      176     171

=== Classification Report ===
              precision    recall  f1-score   support

     비재구매(0)       0.98      0.57      0.73     18304
      재구매(1)       0.02      0.49      0.04       347

    accuracy                           0.57     18651
   macro avg       0.50      0.53      0.38     18651
weighted avg       0.97      0.57      0.71     18651


ROC-AUC: 0.5470


## Step 15. (보완) 이상치 및 다중공선성 확인

모델 해석 전에 두 가지를 점검:
1. 수치형 피처의 분포/이상치 확인 (특히 `delivery_days_clean`, `freight_value`)
2. 피처 간 상관관계 확인 (다중공선성 가정 검증 - `payment_value_total` vs `price`/`item_total`)

In [ ]:
# 1) 수치형 피처 분포 확인
print("=== 수치형 피처 분포 ===")
print(ml_data[numeric_features].describe())

# 2) 다중공선성 확인 - payment_value_total vs price, item_total, freight_value
corr_check = ml_data[['payment_value_total', 'price', 'item_total', 'freight_value']].corr()
print("\n=== 상관관계 ===")
print(corr_check.round(3))

=== 수치형 피처 분포 ===
       delivery_days_clean  freight_value  payment_value_total  \
count         91915.000000   93252.000000         93252.000000   
mean             12.165327      22.774453           160.273911   
std               9.611255      21.557327           220.368375   
min               0.000000       0.000000             9.590000   
25%               6.000000      13.850000            62.000000   
50%              10.000000      17.170000           105.320000   
75%              15.000000      24.000000           176.592500   
max             209.000000    1794.960000         13664.080000   

       payment_installments_max       n_items  review_score_mean  
count              93252.000000  93252.000000       92636.000000  
mean                   2.915112      1.139815           4.153920  
std                    2.701378      0.533980           1.284712  
min                    0.000000      1.000000           1.000000  
25%                    1.000000      1.000000       

## Step 16. 모델 재평가 (RobustScaler 적용)

이상치에 강건한 `RobustScaler` 적용 후 성능 재확인.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("=== Confusion Matrix ===")
print("        예측 0   예측 1")
cm = confusion_matrix(y_test, y_pred)
print(f"실제 0   {cm[0][0]:6d}  {cm[0][1]:6d}")
print(f"실제 1   {cm[1][0]:6d}  {cm[1][1]:6d}")
print()
print("=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=['비재구매(0)', '재구매(1)']))
print()
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")

=== Confusion Matrix ===
        예측 0   예측 1
실제 0    10496    7808
실제 1      176     171

=== Classification Report ===
              precision    recall  f1-score   support

     비재구매(0)       0.98      0.57      0.72     18304
      재구매(1)       0.02      0.49      0.04       347

    accuracy                           0.57     18651
   macro avg       0.50      0.53      0.38     18651
weighted avg       0.97      0.57      0.71     18651


ROC-AUC: 0.5467


## Step 17. 결과 평가 및 다음 단계

`RobustScaler` 적용 후에도 ROC-AUC 0.5467 (이전 0.5470)로 거의 변화 없음.
→ 이상치/스케일링 문제가 아니라, **첫 주문 피처 자체가 재구매를 예측하기에 정보량이 부족**함을 시사.

### 해석
- 재구매율 1.86%의 극희소 이벤트는 첫 주문의 배송/리뷰/결제 정보만으로 예측하기 어려움
- 재구매는 가격 민감도, 재구매 필요 시점, 경쟁사 이용 등 **첫 주문 데이터 밖의 요인**에 더 크게 좌우될 가능성

### 방향 전환
예측 모델로서는 한계가 있으나, **로지스틱 회귀 계수(coefficient)의 방향성**은
"어떤 피처가 재구매와 양/음의 관계인지" 보여줄 수 있음 → 페르소나 도출에 활용

In [ ]:
import pandas as pd

# 전처리 후 피처 이름 + 계수 추출
feature_names = model.named_steps['preprocessor'].get_feature_names_out()
coefs = model.named_steps['classifier'].coef_[0]

coef_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefs
}).sort_values('coefficient', ascending=False)

print("=== 재구매에 긍정적 영향 (상위 10개) ===")
print(coef_df.head(10).to_string(index=False))

print("\n=== 재구매에 부정적 영향 (하위 10개) ===")
print(coef_df.tail(10).to_string(index=False))

=== 재구매에 긍정적 영향 (상위 10개) ===
                                                             feature  coefficient
            cat__product_category_name_english_arts_and_craftmanship     2.223442
                       cat__product_category_name_english_la_cuisine     1.609546
cat__product_category_name_english_furniture_mattress_and_upholstery     1.255285
          cat__product_category_name_english_fashion_underwear_beach     1.218123
         cat__product_category_name_english_costruction_tools_garden     1.184918
         cat__product_category_name_english_fashion_bags_accessories     1.150531
               cat__product_category_name_english_christmas_supplies     1.137004
                       cat__product_category_name_english_cine_photo     1.086776
                           cat__product_category_name_english_drinks     0.977504
                    cat__product_category_name_english_fashion_shoes     0.966102

=== 재구매에 부정적 영향 (하위 10개) ===
                                       

## Step 18. (검증) 극단적 계수의 신뢰성 확인

상위/하위 계수가 모두 `product_category_name_english`에 집중됨.
→ 카테고리별 표본 수가 너무 적어 계수가 불안정할 가능성 (과적합)

확인할 것:
1. 극단적 계수를 보인 카테고리들의 실제 표본 수
2. 핵심 피처(수치형/이진형/지역)의 계수 - 원래 보고자 했던 정보

In [ ]:
# 1) 극단적 계수를 보인 카테고리들의 표본 수 확인
extreme_categories = ['arts_and_craftmanship', 'la_cuisine', 'books_technical', 'home_appliances_2',
                       'agro_industry_and_commerce', 'computers', 'costruction_tools_tools',
                       'furniture_bedroom', 'tablets_printing_image', 'small_appliances_home_oven_and_coffee']

print("=== 극단적 계수 카테고리의 표본 수 ===")
cat_counts = ml_data['product_category_name_english'].value_counts()
print(cat_counts.loc[cat_counts.index.isin(extreme_categories)])

# 2) 핵심 피처(수치형/이진형) 계수만 확인
core_mask = coef_df['feature'].str.startswith('num__') | coef_df['feature'].str.startswith('bin__')
print("\n=== 핵심 피처(수치형/이진형) 계수 ===")
print(coef_df[core_mask].to_string(index=False))

# 3) 지역 피처 계수
print("\n=== 지역(customer_region/seller_region) 계수 ===")
print(coef_df[coef_df['feature'].str.contains('region')].to_string(index=False))

=== 극단적 계수 카테고리의 표본 수 ===
product_category_name_english
books_technical                          252
home_appliances_2                        216
agro_industry_and_commerce               173
computers                                172
costruction_tools_tools                   94
furniture_bedroom                         83
tablets_printing_image                    73
small_appliances_home_oven_and_coffee     72
arts_and_craftmanship                     20
la_cuisine                                12
Name: count, dtype: int64

=== 핵심 피처(수치형/이진형) 계수 ===
                      feature  coefficient
                 num__n_items     0.252768
num__payment_installments_max     0.157662
       num__review_score_mean     0.071474
     num__delivery_days_clean     0.053952
              bin__has_review     0.015426
     num__payment_value_total    -0.031834
           num__freight_value    -0.034054
                 bin__is_late    -0.214386

=== 지역(customer_region/seller_region) 계수 ===
        

In [ ]:
# 지역별 표본 수 확인
print("=== customer_region 분포 ===")
print(ml_data['customer_region'].value_counts())
print("\n=== seller_region 분포 ===")
print(ml_data['seller_region'].value_counts())

=== customer_region 분포 ===
customer_region
남동부    63907
남부     13366
북동부     8797
중서부     5443
북부      1739
Name: count, dtype: int64

=== seller_region 분포 ===
seller_region
남동부    77865
남부     12481
북동부     1485
중서부     1398
북부        23
Name: count, dtype: int64


## Step 19. 모델 단순화 - product_category 제외

`product_category_name_english`는 카디널리티가 높고(70+ 카테고리),
일부 카테고리는 표본 수가 12~20건에 불과해 계수가 불안정함.
→ 핵심 스토리(배송 지연/지역)와도 무관하므로 제외하고 재학습.

`seller_region_북부`(n=23)는 여전히 작지만, 다른 지역들은 충분하므로 유지.

In [ ]:
# product_category_name_english 제외 (고카디널리티 + 소규모 카테고리로 계수 불안정)
categorical_features_v2 = ['customer_region', 'seller_region']

preprocessor_v2 = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features_v2),
    ('bin', 'passthrough', binary_features)
])

model_v2 = Pipeline([
    ('preprocessor', preprocessor_v2),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])

model_v2.fit(X_train, y_train)

y_pred_v2 = model_v2.predict(X_test)
y_proba_v2 = model_v2.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_v2, target_names=['비재구매(0)', '재구매(1)']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba_v2):.4f}")

print("\n=== 계수 (전체) ===")
feature_names_v2 = model_v2.named_steps['preprocessor'].get_feature_names_out()
coefs_v2 = model_v2.named_steps['classifier'].coef_[0]
coef_df_v2 = pd.DataFrame({'feature': feature_names_v2, 'coefficient': coefs_v2}).sort_values('coefficient', ascending=False)
print(coef_df_v2.to_string(index=False))

              precision    recall  f1-score   support

     비재구매(0)       0.98      0.51      0.67     18304
      재구매(1)       0.02      0.54      0.04       347

    accuracy                           0.51     18651
   macro avg       0.50      0.53      0.36     18651
weighted avg       0.97      0.51      0.66     18651

ROC-AUC: 0.5396

=== 계수 (전체) ===
                      feature  coefficient
                 num__n_items     0.267997
num__payment_installments_max     0.164844
     cat__customer_region_남동부     0.135958
        cat__seller_region_남부     0.107694
     cat__customer_region_중서부     0.085746
       num__review_score_mean     0.074984
       cat__seller_region_남동부     0.065885
     num__delivery_days_clean     0.061236
       cat__seller_region_북동부    -0.016671
      cat__customer_region_북부    -0.026325
              bin__has_review    -0.029750
           num__freight_value    -0.030047
        cat__seller_region_북부    -0.040813
     num__payment_value_total    -0.06

## Step 20. 최종 정리

### 모델 성능의 한계
- ROC-AUC 0.54 → 첫 주문 정보만으로 재구매를 예측하기는 어려움
- 재구매율 1.86%의 극희소 이벤트는 가격민감도, 재구매 시점, 경쟁사 이용 등
  첫 주문 데이터 밖의 요인에 더 크게 좌우될 것으로 추정

### 일관되게 확인된 방향성 (두 모델 모두 동일 부호, 충분한 표본)
| 피처 | 계수 | 해석 |
|---|---|---|
| `is_late` | 약 -0.21 | 첫 주문 지연 경험 → 재구매 가능성 ↓ (전체 스토리와 일치) |
| `customer_region_북동부` | 약 -0.23 | 북동부 고객 → 재구매 가능성 ↓ (북동부 우선 개선 근거 추가) |
| `n_items`, `payment_installments_max` | 양수 | 다구매/할부 고객 → 재구매 가능성 ↑ |

### 결론
모델의 예측력은 약하지만, 계수 방향은 "지연율 감소가 재구매율 개선에
기여하는 요인 중 하나"라는 가설과 일치한다. 재구매는 여러 요인의 복합 작용이며,
배송 경험은 그중 하나의 일관된(그러나 단독으로는 약한) 요인이다.

## 부록 - 1~3일 지연 평점 수치 차이 확인 (3.23 vs 2.87)

가능한 원인:
1. 아이템 단위 vs 주문 단위 집계 차이
2. 지연 정의 차이 (`is_late` vs `is_delayed`)

In [ ]:
import pandas as pd

# 베이스 파일 로드 + 배송완료 필터 + days_diff 계산
df_check = pd.read_csv('../../../Funnel_Cohort_NG/data/olist_order_item_level.csv')
df_check = df_check[df_check['order_status'] == 'delivered'].copy()
df_check['order_estimated_delivery_date'] = pd.to_datetime(df_check['order_estimated_delivery_date'])
df_check['order_delivered_customer_date'] = pd.to_datetime(df_check['order_delivered_customer_date'])
df_check['days_diff'] = (df_check['order_delivered_customer_date'] - df_check['order_estimated_delivery_date']).dt.days

# 1) 아이템 단위 (기존 계산 방식)
late_1_3_item = df_check[(df_check['days_diff'] > 0) & (df_check['days_diff'] <= 3)]
print(f"아이템 단위: {late_1_3_item['review_score_mean'].mean():.3f}점 (n={len(late_1_3_item)})")

# 2) 주문 단위로 중복 제거 후 재계산
order_dedup = df_check.drop_duplicates(subset='order_id')
late_1_3_order = order_dedup[(order_dedup['days_diff'] > 0) & (order_dedup['days_diff'] <= 3)]
print(f"주문 단위: {late_1_3_order['review_score_mean'].mean():.3f}점 (n={len(late_1_3_order)})")

아이템 단위: 3.231점 (n=2110)
주문 단위: 3.291점 (n=1870)


In [ ]:
# cohort_df 기준으로 재계산 (delay_days, review_score)
# - delay_days: anomaly_flag 필터링 없는 별도 파이프라인에서 계산
# - review_score: review_score_mean이 아닌 원본 리뷰 점수 (review_score)
cohort_check = cohort_df[cohort_df['is_valid_funnel'] == True].copy()

late_1_3_cohort = cohort_check[(cohort_check['delay_days'] > 0) & (cohort_check['delay_days'] <= 3)]
print(f"cohort_df 기준 (delay_days+review_score): {late_1_3_cohort['review_score'].mean():.3f}점 (n={len(late_1_3_cohort)})")

# delay_days를 정수로 올림 처리한 경우도 확인 (소수점 days 처리 차이 가능성)
import numpy as np
cohort_check['delay_days_ceil'] = np.ceil(cohort_check['delay_days'])
late_1_3_ceil = cohort_check[(cohort_check['delay_days_ceil'] > 0) & (cohort_check['delay_days_ceil'] <= 3)]
print(f"cohort_df 기준 (delay_days 올림, 1~3일): {late_1_3_ceil['review_score'].mean():.3f}점 (n={len(late_1_3_ceil)})")

cohort_df 기준 (delay_days+review_score): 3.767점 (n=2644)
cohort_df 기준 (delay_days 올림, 1~3일): 3.767점 (n=2644)


In [ ]:
# '정시'를 days_diff == 0(정확히 예정일과 일치)으로만 한정
exact_on_time = df_check[df_check['days_diff'] == 0]
early = df_check[df_check['days_diff'] < 0]

print(f"정시(==0): {exact_on_time['review_score_mean'].mean():.3f}점 (n={len(exact_on_time)})")
print(f"조기(<0): {early['review_score_mean'].mean():.3f}점 (n={len(early)})")

정시(==0): 3.987점 (n=1450)
조기(<0): 4.212점 (n=101475)


## 부록 - 1~3일 지연 평점 수치 차이 확인 (3.23 vs 2.87)

### 배경
분석(`ab_test_eta.ipynb`)에서 1~3일 지연 그룹 평균 리뷰는 **3.23점**(n=2,110)이었으나,
다른 시각화에서는 동일 구간이 **2.87점**으로 나타나 수치 차이 확인이 필요했음.
("정시" 기준값도 4.21점 vs 3.68점으로 차이 존재)

### 확인한 가설과 결과

| # | 가설 | 방법 | 결과 |
|---|---|---|---|
| 1 | 아이템 단위 vs 주문 단위 | `drop_duplicates(subset='order_id')` 후 재계산 | 3.231 → 3.291 (오히려 증가, 가설 기각) |
| 2 | `is_late`(base_df) vs `is_delayed`(cohort_df) 정의 차이 | `cohort_df`의 `delay_days`+`review_score`로 재계산 | 3.767 (오히려 증가, 가설 기각) |
| 3 | delay_days 소수점 처리(올림) 차이 | `np.ceil(delay_days)` 적용 | 3.767 (동일, 영향 없음) |
| 4 | "정시" 정의 차이 (조기 포함 여부) | `days_diff == 0`만으로 "정시" 재정의 | 3.987 (3.68과 여전히 0.3점 차이) |

### 결론
4가지 가설 모두 2.87/3.68에 도달하지 못함. 

### 결정
- 분석 내에서는 자체 파이프라인(`is_late`, `days_diff`, `review_score_mean`) 기준
  **3.23점(1~3일 지연) / 4.21점(정시·조기)**을 일관되게 사용